# Clinic patient data cleaning

Adapted from your retail experiment. Place this notebook and `clinic_patients_dirty.csv` in the same folder. Select a Python kernel with pandas and numpy installed, then choose **Run All**.

The input stays unchanged. Suspicious values are flagged, clinical measurements are not imputed, and repeated patient IDs are reviewed rather than automatically deleted. Screening bounds are classroom checks, not diagnostic thresholds.

## 1. Define the cleaning workflow

The following cell contains the complete workflow, with comments for each step: tidy headers; expose missing tokens; remove duplicates; standardise labels; parse dates; separate paired blood pressure; convert explicit units; flag ambiguous and suspicious values; compare BMI; validate; export.

In [5]:
!wget https://raw.githubusercontent.com/emmanuelyardamah/Data-Warehouse-and-Data-Mining/refs/heads/main/clinic_data_cleaning.ipynb

--2026-10-11 12:28:04--  https://raw.githubusercontent.com/emmanuelyardamah/Data-Warehouse-and-Data-Mining/refs/heads/main/clinic_data_cleaning.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16685 (16K) [text/plain]
Saving to: ‘clinic_data_cleaning.ipynb.2’

clinic_data_cleanin 100%[===================>]  16.29K  --.-KB/s    in 0.001s  

2026-10-11 12:28:04 (28.8 MB/s) - ‘clinic_data_cleaning.ipynb.2’ saved [16685/16685]



In [6]:
"""Clinic dataset cleaning. Requires pandas and numpy. Original CSV is never changed."""
from pathlib import Path
import re
import numpy as np
import pandas as pd


def clean_clinic(input_path="clinic_patients_dirty.csv", output_dir="cleaning_output"):
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    raw = pd.read_csv(input_path, dtype=str, keep_default_na=False)
    df = raw.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"\(.*?\)", "", regex=True).str.strip()
                  .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
    df = df.rename(columns={"height": "height_cm", "weight": "weight_kg", "temp": "temperature_c"})
    required = {"patient_id", "full_name", "sex", "age", "county", "visit_date", "height_cm", "weight_kg", "bmi", "systolic_bp", "diastolic_bp", "temperature_c", "blood_group", "diagnosis", "smoker"}
    if set(df.columns) != required:
        raise ValueError(f"Unexpected columns. Missing: {required - set(df.columns)}; extra: {set(df.columns) - required}")
    df = df.apply(lambda s: s.str.strip().str.replace(r"\s+", " ", regex=True))
    tokens = {"", "n/a", "na", "null", "none", "nan", "?", "-", "unknown", "not recorded"}
    df = df.mask(df.apply(lambda s: s.str.lower().isin(tokens)))
    missing_before = df.isna().sum()
    df["patient_id"] = df["patient_id"].str.upper().str.replace(r"[\s_]+", "-", regex=True)
    duplicate_mask = df.duplicated()
    removed = df.loc[duplicate_mask].copy()
    removed.insert(0, "source_row", removed.index + 2)
    removed.to_csv(output / "removed_duplicates.csv", index=False)
    df = df.loc[~duplicate_mask].copy()
    df.insert(0, "source_row", df.index + 2)
    original = df.copy()
    issues = []

    def flag(mask, column, reason):
        for idx in df.index[pd.Series(mask, index=df.index).fillna(False)]:
            issues.append({"source_row": int(df.at[idx, "source_row"]),
                           "patient_id": df.at[idx, "patient_id"], "column": column,
                           "original_value": original.at[idx, column], "reason": reason})

    def map_labels(column, mapping):
        txt = df[column].str.lower()
        unknown = txt.notna() & ~txt.isin(mapping)
        flag(unknown, column, "Unrecognised label retained; review mapping")
        df[column] = txt.map(mapping).where(~unknown, df[column])

    map_labels("sex", {"m": "Male", "male": "Male", "f": "Female", "female": "Female"})
    map_labels("smoker", {**dict.fromkeys(["y", "yes", "true", "1"], "Yes"),
                          **dict.fromkeys(["n", "no", "false", "0"], "No")})
    df["full_name"] = df["full_name"].str.title()
    county = df["county"].str.lower().str.replace(r"\s+county$", "", regex=True).str.rstrip(".")
    county = county.replace({"nairobbi": "nairobi", "nrb": "nairobi", "mombassa": "mombasa", "kiambuu": "kiambu", "machackos": "machakos"})
    df["county"] = county.str.title()
    aliases = {"maleria": "malaria", "hypertention": "hypertension", "htn": "hypertension", "high blood pressure": "hypertension", "t2dm": "type 2 diabetes", "type ii diabetes": "type 2 diabetes", "diabetes type 2": "type 2 diabetes", "urti": "upper respiratory tract infection", "upper resp. tract infection": "upper respiratory tract infection", "pnuemonia": "pneumonia", "typhoid fever": "typhoid"}
    df["diagnosis"] = df["diagnosis"].str.lower().replace(aliases).str.title()
    blood = df["blood_group"].str.upper().str.replace("POSITIVE", "+", regex=False).str.replace("NEGATIVE", "-", regex=False).str.replace("POS", "+", regex=False).str.replace("NEG", "-", regex=False).str.replace(" ", "", regex=False)
    flag(blood.notna() & ~blood.isin(["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]), "blood_group", "Unrecognised blood group retained")
    df["blood_group"] = blood

    # Explicit date formats: day/month/year for slash and dotted formats.
    parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%d.%m.%Y", "%d-%b-%Y", "%B %d, %Y"]:
        parsed = parsed.fillna(pd.to_datetime(df["visit_date"], format=fmt, errors="coerce"))
    flag(df["visit_date"].notna() & parsed.isna(), "visit_date", "Invalid or unsupported date; set to missing")
    df["visit_date"] = parsed

    # Recover paired BP only when it does not conflict with a separate value.
    pair = df["systolic_bp"].str.extract(r"^(\d+(?:\.\d+)?)/(\d+(?:\.\d+)?)$")
    paired = pair[0].notna()
    separate = pd.to_numeric(df["diastolic_bp"], errors="coerce")
    conflict = paired & separate.notna() & separate.ne(pd.to_numeric(pair[1], errors="coerce"))
    flag(conflict, "systolic_bp", "Paired BP conflicts with separate diastolic value; separate value retained")
    recover = paired & df["diastolic_bp"].isna()
    df.loc[recover, "diastolic_bp"] = pair.loc[recover, 1]
    df.loc[paired, "systolic_bp"] = pair.loc[paired, 0]

    def numeric(column, suffix=""):
        txt = df[column].str.lower()
        if suffix:
            txt = txt.str.replace(suffix, "", regex=True).str.strip()
        result = pd.to_numeric(txt, errors="coerce")
        flag(df[column].notna() & result.isna(), column, "Could not parse number; set to missing")
        return result

    df["age"] = numeric("age", r"\s*(?:years?|yrs?)$")
    flag(df["age"].notna() & ((df["age"] < 0) | (df["age"] > 120) | (df["age"] % 1 != 0)), "age", "Outside teaching dataset age rule (0-120 whole years); retained for review")
    for col, suffix in [("weight_kg", r"\s*kg$"), ("bmi", ""), ("systolic_bp", ""), ("diastolic_bp", "")]:
        df[col] = numeric(col, suffix)
        invalid = df[col].notna() & (df[col] <= 0)
        flag(invalid, col, "Nonpositive measurement; set to missing")
        df.loc[invalid, col] = np.nan

    height_text = df["height_cm"].str.lower()
    metres = height_text.str.contains(r"m$", na=False) & ~height_text.str.contains(r"cm$", na=False)
    height = numeric("height_cm", r"\s*(?:cm|m)$")
    # Small unlabelled heights remain ambiguous rather than silently assuming metres.
    df["height_cm"] = height.where(~metres, height * 100)
    ambiguous_height = ~metres & ~height_text.str.contains("cm", na=False) & height.between(0, 3, inclusive="neither")
    flag(ambiguous_height, "height_cm", "Unlabelled small height: possible metres; set to missing pending confirmation")
    invalid_height = df["height_cm"].notna() & (df["height_cm"] <= 0)
    flag(invalid_height, "height_cm", "Nonpositive height; set to missing")
    df.loc[ambiguous_height | invalid_height, "height_cm"] = np.nan

    temp_text = df["temperature_c"].str.upper()
    fahrenheit = temp_text.str.contains(r"F$", na=False)
    explicit_c = temp_text.str.contains(r"C$", na=False)
    temperature = pd.to_numeric(temp_text.str.replace(r"\s*[°º]?\s*[CF]$", "", regex=True).str.strip(), errors="coerce")
    flag(temp_text.notna() & temperature.isna(), "temperature_c", "Could not parse temperature; set to missing")
    df["temperature_c"] = temperature.where(~fahrenheit, (temperature - 32) * 5 / 9).round(2)
    ambiguous_temp = ~fahrenheit & ~explicit_c & temperature.gt(60)
    flag(ambiguous_temp, "temperature_c", "Unlabelled high temperature: possible Fahrenheit; set to missing pending confirmation")
    invalid_temp = df["temperature_c"].notna() & (df["temperature_c"] <= 0)
    flag(invalid_temp, "temperature_c", "Nonpositive temperature; set to missing")
    df.loc[ambiguous_temp | invalid_temp, "temperature_c"] = np.nan

    # Broad classroom screening bounds, not clinical diagnostic thresholds.
    for col, bounds in {"height_cm": (50, 250), "weight_kg": (2, 350), "bmi": (5, 100), "systolic_bp": (40, 300), "diastolic_bp": (20, 200), "temperature_c": (30, 45)}.items():
        flag(df[col].notna() & ~df[col].between(*bounds), col, f"Outside broad classroom screening range {bounds}; retained for review")
    flag(df["systolic_bp"].notna() & df["diastolic_bp"].notna() & df["systolic_bp"].le(df["diastolic_bp"]), "systolic_bp", "Systolic is not above diastolic; review")
    valid_body = df["height_cm"].between(50, 250) & df["weight_kg"].between(2, 350)
    df["bmi_calculated"] = (df["weight_kg"] / (df["height_cm"] / 100) ** 2).where(valid_body).round(1)
    flag(df["bmi"].notna() & df["bmi_calculated"].notna() & (df["bmi"] - df["bmi_calculated"]).abs().gt(0.2), "bmi", "Recorded BMI differs from calculated BMI by more than 0.2; both preserved")
    flag(df["patient_id"].isna() | ~df["patient_id"].str.fullmatch(r"PT-\d{4}", na=False), "patient_id", "Missing or malformed patient ID")
    flag(df["patient_id"].notna() & df["patient_id"].duplicated(keep=False), "patient_id", "Repeated patient ID: review visit/record context; retained")
    review = pd.DataFrame(issues, columns=["source_row", "patient_id", "column", "original_value", "reason"])
    df["needs_review"] = df["source_row"].isin(review["source_row"])
    report = pd.DataFrame({"missing_before": original[list(required)].isna().sum(), "missing_after": df[list(required)].isna().sum()})
    audit = []
    for col in sorted(required):
        after = df[col].astype("string").fillna("<missing>")
        before = original[col].astype("string").fillna("<missing>")
        for idx in df.index[after.ne(before)]:
            audit.append({"source_row": int(df.at[idx, "source_row"]), "column": col, "before": before.at[idx], "after": after.at[idx]})
    assert len(raw) == len(df) + len(removed)
    assert df["source_row"].is_unique
    assert pd.api.types.is_datetime64_any_dtype(df["visit_date"])
    assert all(pd.api.types.is_numeric_dtype(df[c]) for c in ["age", "height_cm", "weight_kg", "bmi", "systolic_bp", "diastolic_bp", "temperature_c"])
    df.to_csv(output / "clinic_patients_clean.csv", index=False, date_format="%Y-%m-%d")
    review.to_csv(output / "clinic_patients_review.csv", index=False)
    report.to_csv(output / "quality_report.csv", index_label="column")
    pd.DataFrame(audit, columns=["source_row", "column", "before", "after"]).to_csv(output / "cleaning_log.csv", index=False)
    print(f"Rows: {len(raw)} -> {len(df)}; duplicate rows removed: {len(removed)}")
    print(f"Records needing review: {df['needs_review'].sum()}; review issues: {len(review)}")
    print("Validation checks passed. No missing clinical measurements were imputed.")
    print(f"Files saved in: {output.resolve()}")
    print(report.to_string())
    return df, review, report



## 2. Run the workflow

Change `DATA_PATH` only if your input file has another name or location. Results are written to the `cleaning_output` folder.

In [7]:
DATA_PATH = "https://raw.githubusercontent.com/emmanuelyardamah/Data-Warehouse-and-Data-Mining/refs/heads/main/clinic_patients_dirty.csv"
OUT_DIR = "cleaning_output"
df, review, report = clean_clinic(DATA_PATH, OUT_DIR)


Rows: 420 -> 400; duplicate rows removed: 20
Records needing review: 81; review issues: 95
Validation checks passed. No missing clinical measurements were imputed.
Files saved in: /content/cleaning_output
               missing_before  missing_after
patient_id                  0              0
smoker                     32             32
bmi                        22             28
weight_kg                  22             22
age                        21             21
systolic_bp                17             21
diastolic_bp               35             19
full_name                   0              0
temperature_c              16             40
blood_group                40             40
county                     20             20
height_cm                  16             34
sex                        16             16
diagnosis                  12             12
visit_date                 12             13


## 3. Inspect the result

Missing values are allowed. `needs_review` means one or more issues need checking against the source. `source_row` points to the original CSV line, counting the header as line 1.

In [8]:
display(df.head(10))
display(df.dtypes.to_frame("data_type"))
display(report)
display(review.head(30))


,source_row,patient_id,full_name,sex,age,county,visit_date,height_cm,weight_kg,bmi,systolic_bp,diastolic_bp,temperature_c,blood_group,diagnosis,smoker,bmi_calculated,needs_review
0,2,PT-0165,Purity Omondi,Female,19.0,Machakos,2025-06-09,164.0,83.8,21.7,142.0,NaN,36.9,O+,Hypertension,NaN,31.2,True
1,3,PT-0350,Halima Barasa,Female,45.0,Kiambu,2025-06-01,160.0,44.5,17.4,119.0,77.0,38.9,B+,Malaria,Yes,17.4,False
2,4,PT-0395,David Otieno,Male,27.0,Nairobi,2025-10-27,177.0,79.1,NaN,143.0,96.0,38.2,O+,Hypertension,No,25.2,False
3,5,PT-0075,Collins Kariuki,Male,28.0,Nairobi,2025-05-29,NaN,54.1,19.6,160.0,98.0,38.0,NaN,Hypertension,No,NaN,False
4,6,PT-0240,Grace Barasa,Female,25.0,Kiambu,2025-03-13,163.0,63.8,24.0,121.0,71.0,37.9,O+,Typhoid,No,24.0,False
5,7,PT-0215,Faith Abdi,Female,18.0,Kiambu,2025-05-22,170.0,49.1,17.0,111.0,69.0,38.2,A+,Pneumonia,No,17.0,False
6,8,PT-0100,Victor Kariuki,Male,40.0,Nakuru,2025-03-23,158.0,66.8,26.8,113.0,76.0,39.1,O+,Pneumonia,No,26.8,False
7,9,PT-0061,Brian Njoroge,Male,58.0,Kisumu,2025-02-24,169.0,79.9,28.0,149.0,98.0,37.9,O+,Hypertension,No,28.0,False
8,10,PT-0200,Halima Omondi,Female,33.0,Nairobi,2025-08-15,166.0,70.8,25.7,134.0,87.0,NaN,B+,Malaria,No,25.7,False
9,11,PT-0229,Brian Barasa,Male,45.0,Kisumu,2025-05-01,163.0,66.6,25.1,102.0,66.0,37.0,A+,Type 2 Diabetes,No,25.1,False


,data_type
source_row,int64
patient_id,object
full_name,object
sex,object
age,float64
county,object
visit_date,datetime64[ns]
height_cm,float64
weight_kg,float64
bmi,float64


,missing_before,missing_after
patient_id,0,0
smoker,32,32
bmi,22,28
weight_kg,22,22
age,21,21
systolic_bp,17,21
diastolic_bp,35,19
full_name,0,0
temperature_c,16,40
blood_group,40,40


,source_row,patient_id,column,original_value,reason
0,167,PT-0185,visit_date,31/06/2025,Invalid or unsupported date; set to missing
1,45,PT-0015,age,999,Outside teaching dataset age rule (0-120 whole...
2,84,PT-0309,age,365,Outside teaching dataset age rule (0-120 whole...
3,123,PT-0361,age,-4,Outside teaching dataset age rule (0-120 whole...
4,126,PT-0107,age,999,Outside teaching dataset age rule (0-120 whole...
5,153,PT-0387,age,999,Outside teaching dataset age rule (0-120 whole...
6,216,PT-0025,age,130,Outside teaching dataset age rule (0-120 whole...
7,317,PT-0315,age,365,Outside teaching dataset age rule (0-120 whole...
8,95,PT-0268,bmi,0,Nonpositive measurement; set to missing
9,102,PT-0084,bmi,0,Nonpositive measurement; set to missing


## 4. Understand the outputs

- `clinic_patients_clean.csv`: standardised data, including original recorded BMI and a separate calculated BMI.
- `clinic_patients_review.csv`: flagged values and reasons.
- `cleaning_log.csv`: changes after initial text/missing-value/ID normalisation.
- `removed_duplicates.csv`: removed records after initial normalisation.
- `quality_report.csv`: missing counts before and after typed cleaning, for the same retained rows.

Dates with slashes or dots use day/month/year. Explicit metre heights become centimetres; explicit Fahrenheit temperatures become Celsius. Unlabelled small heights and unlabelled temperatures above 60 are left missing for confirmation. Other unlabelled heights and temperatures follow the target cm/C convention and are screened. Invalid nonpositive measurements are blanked. Extreme positive measurements remain flagged. BMI is calculated only from measurements within the broad screening bounds. No inference of diagnoses or blood groups is performed.

Duplicate removal compares all fields after whitespace, missing-token and ID normalisation. A repeated patient ID alone does not imply a duplicate visit. Consult the review file before analysis.

## Discussion

1. Why can missing counts increase after cleaning?
2. Why should an outlier remain available for review?
3. What assumptions would be required to convert unlabelled 1.62 heights or 98.6 temperatures?
4. Why preserve recorded and calculated BMI separately?